# ML-07 -- Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ak470107/ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Lane 2: Refresh / Content Opportunity Scoring. This notebook does three things in order: check two signals my rule leans on (with bucket tables and n), encode one transparent rule as a score + one reason code + an action label and write the ranked queue, then hand-review the top ten with a skeptic's eye.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `building-baselines` + `flyrank/flyrank-data` for this task.

## 1. Two signal checks first

*Pick two signals my rule idea leans on. At least one has to sit behind a real FlyRank flag from the session.*

**Signal A -- staleness, behind the refresh flags.** The claim: "a page that hasn't been updated in a while is more likely to be declining" -- this is the premise every refresh flag rests on. I bucket by `freshness_tier` (built from `days_since_last_update`) and compare `is_declining_label` mean (the decline rate) and n per bucket.

**Signal B -- CTR vs. position, behind the CTR-fix logic.** The claim: "a better search position should mean a higher click-through rate" -- this is the premise behind flagging a page for a title/meta rewrite instead of a content refresh. I bucket by `position_tier` (built from `avg_position`) and compare CTR (both the plain per-page mean and the click-weighted rate, since traffic is heavy-tailed) and n per bucket.

Both signals load the same starter dataset used in w01-w03 (`data/raw/content_refresh_anonymized.csv`, one row per content item). No warehouse query needed for a rule this simple -- everything here is already in the 44-column CSV.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import os

local_path = "../../data/raw/content_refresh_anonymized.csv"

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
elif os.path.exists("ML-internship/data/raw/content_refresh_anonymized.csv"):
    df = pd.read_csv("ML-internship/data/raw/content_refresh_anonymized.csv")
else:
    # Colab fallback: clone the repo, then read from it
    !git clone --depth 1 https://github.com/ak470107/ML-internship.git
    df = pd.read_csv("ML-internship/data/raw/content_refresh_anonymized.csv")

# Derive the label myself for the SIGNAL CHECK ONLY -- it is never used inside the rule/score below.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Rows, columns:", df.shape)
print("One row =", df["content_id"].nunique(), "unique content pages")

Cloning into 'ML-internship'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 91 (delta 11), reused 62 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 1.86 MiB | 14.22 MiB/s, done.
Resolving deltas: 100% (11/11), done.
Rows, columns: (30000, 45)
One row = 30000 unique content pages


### Signal A -- staleness vs. decline rate, by `freshness_tier`

Bucket table with n and decline rate per bucket:

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

signal_a = (
    df.groupby("freshness_tier")["is_declining_label"]
      .agg(n="size", decline_rate="mean")
      .reindex(["0-30", "31-90", "91-180", "181+"])  # youngest -> oldest, on purpose
)
signal_a["decline_rate_pct"] = (signal_a["decline_rate"] * 100).round(1)
print(signal_a[["n", "decline_rate_pct"]])

                    n  decline_rate_pct
freshness_tier                         
0-30            20480              51.1
31-90             175              58.9
91-180           9171              61.1
181+              174              47.1


**Verdict: MIXED.** The direction holds for most of the range -- decline rate climbs from 51.1% (`0-30`, n=20,480) to 58.9% (`31-90`, n=175) to a peak of 61.1% (`91-180`, n=9,171). But it does **not** keep climbing: the oldest bucket, `181+` (n=174), drops back to 47.1% -- *below* the freshest bucket. That reversal sits in the two thinnest buckets in the whole table (174-175 rows each, versus 20,480 and 9,171 for the other two), so it's a real but fragile finding, not noise from a single row. The honest reading: staleness predicts decline cleanly up to about six months out, then the signal breaks down for the truly ancient pages -- probably because a page that's survived 181+ days untouched without dying already found a stable audience that doesn't need a rescue. That reversal is exactly why my rule below only flags the `91-180` tier as "stale," not "the older the better."

**Signal-check honesty:** the label itself (`is_declining_label` / `trend_direction`) is used here only to *test* the staleness claim -- it never enters the score in Section 2.

### Signal B -- CTR vs. position, by `position_tier`

Bucket table with n, the plain per-page mean CTR, and the click-weighted CTR (clicks summed / impressions summed, since a mean of per-page rates is dominated by the same giants the data dictionary warns about):

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

tier_order = ["deep", "page_3_5", "striking", "page_1", "top_3"]  # worst position -> best position
signal_b = df.groupby("position_tier").agg(
    n=("content_id", "size"),
    clicks=("clicks_90d", "sum"),
    impressions=("impressions_90d", "sum"),
    mean_ctr=("ctr", "mean"),
).reindex(tier_order)
signal_b["weighted_ctr"] = (signal_b["clicks"] / signal_b["impressions"] * 100).round(3)
signal_b["mean_ctr"] = signal_b["mean_ctr"].round(3)
print(signal_b[["n", "mean_ctr", "weighted_ctr"]])

                   n  mean_ctr  weighted_ctr
position_tier                               
deep            1319     0.150         0.041
page_3_5        7242     0.222         0.155
striking        7304     0.323         0.347
page_1         11814     0.652         0.350
top_3           2321     1.484         0.488


**Verdict: CONFIRMED, with a caveat.** The plain per-page mean CTR is cleanly monotonic in the direction the CTR-fix logic assumes: `deep` (0.15%) < `page_3_5` (0.22%) < `striking` (0.32%) < `page_1` (0.65%) < `top_3` (1.48%), across five buckets of at least 1,300 rows each. The click-weighted rate confirms the same story at the extremes (`deep` worst, `top_3` best by a wide margin) but is nearly flat in the middle -- `page_1` (0.350%) and `striking` (0.347%) come out almost tied once a handful of very-high-volume pages dominate each bucket's sum. That's the heavy-tail warning from `flyrank-data` showing up directly: a few giant pages in the `page_1`/`striking` buckets pull the weighted number toward their own CTR and away from what a "typical" page in that tier looks like. Position and CTR do move together -- I'm using the plain per-page mean as the reference for the rule below, since the rule scores one page at a time and the weighted number answers a different question ("what's the platform's overall CTR"), not this one.

## 2. My rule, in plain words -- then the ranked queue

**The rule, in three sentences:** A page is worth a refresh review if it sat in the `91-180` freshness tier -- the one bucket where staleness actually predicts decline (Signal A) -- and it still has enough real search visibility to be worth anyone's time (`impressions_90d >= 300`, the same "moderate" floor the `impression_tier` column already uses). Among those pages, the ones ranking well already (`top_3`/`page_1`/`striking`) but pulling a below-tier-average CTR (Signal B) get flagged as a **CTR fix** -- rewrite the title/meta, not the body. Everyone else who clears both bars gets flagged as a plain **stale refresh** -- the content itself needs updating. The score inside each bucket is just `impressions_90d`: bigger current audience, bigger payoff from getting it right, no fitted weights.

**Reason codes this rule can output (one per row):**
- `ctr_fix_opportunity` -- stale, visible, ranks fine, CTR lags its own tier's average
- `stale_needs_refresh` -- stale, visible, everything else
- `not_flagged` -- score is 0; fails the staleness or visibility bar

**Action labels (one per reason code):** `optimize_title_meta`, `refresh_content`, `no_action`.

No label-derived or future-window column enters the score or the reason-code logic -- only `freshness_tier`, `impressions_90d`, `position_tier`, and `ctr`, all knowable the day someone looks at the page.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

# 1) stale: the one freshness_tier where Signal A actually held up
stale = (df["freshness_tier"] == "91-180").astype(int)

# 2) visible: same "moderate" floor impression_tier already uses (>= 300 impressions/90d)
visible = (df["impressions_90d"] >= 300).astype(int)

# 3) score: transparent, readable on purpose -- impressions_90d, gated by the two flags
df["score"] = stale * visible * df["impressions_90d"]

# 4) reason code: CTR-fix vs. plain stale refresh, using Signal B's per-tier mean CTR as the bar
tier_mean_ctr = df.groupby("position_tier")["ctr"].transform("mean")
good_position = df["position_tier"].isin(["top_3", "page_1", "striking"])
ctr_lags_tier = df["ctr"] < tier_mean_ctr

flagged = (stale == 1) & (visible == 1)
df["reason_code"] = np.select(
    [flagged & good_position & ctr_lags_tier, flagged],
    ["ctr_fix_opportunity", "stale_needs_refresh"],
    default="not_flagged",
)
df["action"] = df["reason_code"].map({
    "ctr_fix_opportunity": "optimize_title_meta",
    "stale_needs_refresh": "refresh_content",
    "not_flagged": "no_action",
})

print(df["reason_code"].value_counts())
print()
print(df["action"].value_counts())

reason_code
not_flagged            22788
ctr_fix_opportunity     3901
stale_needs_refresh     3311
Name: count, dtype: int64

action
no_action              22788
optimize_title_meta     3901
refresh_content         3311
Name: count, dtype: int64


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

os.makedirs("../outputs", exist_ok=True)

queue = (
    df[df["score"] > 0]
    .sort_values("score", ascending=False)
    [["content_id", "client_id", "score", "reason_code", "action",
      "impressions_90d", "avg_position", "position_tier", "ctr", "freshness_tier"]]
    .reset_index(drop=True)
)

out_path = "../outputs/baseline_action_score.csv"
queue.to_csv(out_path, index=False)

print(f"Ranked queue: {len(queue):,} flagged pages out of {len(df):,} total ({len(queue)/len(df):.1%})")
print(f"Written to {out_path}")
queue.head(10)

Ranked queue: 7,212 flagged pages out of 30,000 total (24.0%)
Written to ../outputs/baseline_action_score.csv


,content_id,client_id,score,reason_code,action,impressions_90d,avg_position,position_tier,ctr,freshness_tier
0,content_5fe46e04994d,client_4e07408562,517715,ctr_fix_opportunity,optimize_title_meta,517715,4.2,page_1,0.14,91-180
1,content_2dba2b1f9536,client_6208ef0f77,443434,stale_needs_refresh,refresh_content,443434,27.9,page_3_5,0.21,91-180
2,content_2c2606c5d176,client_19581e27de,347399,ctr_fix_opportunity,optimize_title_meta,347399,4.2,page_1,0.53,91-180
3,content_cb112fce36be,client_19581e27de,309910,ctr_fix_opportunity,optimize_title_meta,309910,5.6,page_1,0.16,91-180
4,content_9532f197bbc8,client_4e07408562,309192,ctr_fix_opportunity,optimize_title_meta,309192,2.0,top_3,0.87,91-180
5,content_36ff89c8214e,client_19581e27de,295097,ctr_fix_opportunity,optimize_title_meta,295097,7.3,page_1,0.05,91-180
6,content_b28d1efd668f,client_6208ef0f77,286608,stale_needs_refresh,refresh_content,286608,26.2,page_3_5,0.06,91-180
7,content_813e88069237,client_6208ef0f77,233561,stale_needs_refresh,refresh_content,233561,26.2,page_3_5,0.06,91-180
8,content_c21024970297,client_19581e27de,211366,ctr_fix_opportunity,optimize_title_meta,211366,5.1,page_1,0.41,91-180
9,content_c8e9d6ab9013,client_19581e27de,208678,ctr_fix_opportunity,optimize_title_meta,208678,9.7,page_1,0.00,91-180


## 3. Top-10 review

One line each: the action, why it's there, and what would make it wrong.

1. **`content_5fe46e04994d`** -- score 517,715. **Action:** `optimize_title_meta`. **Why:** stale (91-180 tier), 517,715 impressions, ranks `page_1` (avg position 4.2) but pulls only 0.14% CTR against a page_1 average near 0.65% -- ranking well, badly under-clicked. **What would make it wrong:** if the page 1 average CTR I compare against is itself inflated by a few giant pages (the Signal B weighted-vs-mean gap above) -- then 0.14% might not actually be a below-tier outlier.
2. **`content_2dba2b1f9536`** -- score 443,434. **Action:** `refresh_content`. **Why:** stale, largest score in the queue, but only `page_3_5` (avg position 27.9) with 0.21% CTR -- position is the bigger problem than the click-through rate here. **What would make it wrong:** `is_declining_label` for this page is actually 0 (not declining) -- a real content manager should sanity-check this one before spending three hours on it.
3. **`content_2c2606c5d176`** -- score 347,399. **Action:** `optimize_title_meta`. **Why:** same pattern as #1 -- stale, visible, `page_1` (4.2), CTR 0.53% still under its tier's ~0.65% average. **What would make it wrong:** 0.53% is close to the tier average, not far below it -- a tighter CTR-gap threshold (e.g. 20% below tier mean, not any amount below) would likely drop this one.
4. **`content_cb112fce36be`** -- score 309,910. **Action:** `optimize_title_meta`. **Why:** stale, `page_1` (5.6), CTR 0.16% -- clearly under-clicked for the position. **What would make it wrong:** same client (`client_19581e27de`) as three other rows in this top 10 -- if that client's tracking pixel or SERP snippet is broken sitewide, this is one bug, not four separate opportunities.
5. **`content_9532f197bbc8`** -- score 309,192. **Action:** `optimize_title_meta`. **Why:** stale, ranks `top_3` (2.0!) -- the best position in the dataset -- yet only 0.87% CTR, below what a `top_3` page should pull. **What would make it wrong:** `top_3` pages have a small median volume per the data dictionary's own warning -- if this page's 0.87% is based on very few impressions, one page could be swinging the whole `top_3` tier average I'm comparing it against.
6. **`content_36ff89c8214e`** -- score 295,097. **Action:** `optimize_title_meta`. **Why:** stale, `page_1` (7.3), CTR 0.05% -- the lowest CTR of anyone in the top 10 at this volume. **What would make it wrong:** `is_declining_label` is 0 here too -- worth checking whether this page recently changed its title/meta already and the CTR hasn't caught up yet in the data.
7. **`content_b28d1efd668f`** -- score 286,608. **Action:** `refresh_content`. **Why:** stale, `page_3_5` (26.2), CTR 0.06% -- weak on both position and CTR, a genuine refresh case rather than a quick title fix. **What would make it wrong:** same client and near-identical position (26.2) as row #9 below -- if this is one template shared across many pages for this client, refreshing one won't move the others.
8. **`content_813e88069237`** -- score 233,561. **Action:** `refresh_content`. **Why:** same client, same position (26.2), CTR 0.06% -- essentially the same profile as #7. **What would make it wrong:** this duplication is itself the flag -- three near-identical rows from one client (see #7, #9) suggests a client-level pattern the rule can't see, not three independent opportunities.
9. **`content_c21024970297`** -- score 211,366. **Action:** `optimize_title_meta`. **Why:** stale, `page_1` (5.1), CTR 0.41% -- moderately under-tier, but the smallest CTR gap of the `ctr_fix_opportunity` rows in this top 10. **What would make it wrong:** at 0.41% vs. a ~0.65% tier average, this is the borderline case -- a stricter gap threshold drops this one first.
10. **`content_c8e9d6ab9013`** -- score 208,678. **Action:** `optimize_title_meta`. **Why:** stale, `page_1` (9.7, right at the edge of page 1), CTR 0.00% -- zero clicks despite real impressions, the starkest CTR gap in the top 10. **What would make it wrong:** 9.7 is on the boundary between `page_1` and `striking` -- if this page's true average position drifts to 10.1 on a slightly different day, it moves tiers and the comparison bar it's judged against changes with it.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

pd.set_option("display.width", 200)
top10 = queue.head(10)
top10

,content_id,client_id,score,reason_code,action,impressions_90d,avg_position,position_tier,ctr,freshness_tier
0,content_5fe46e04994d,client_4e07408562,517715,ctr_fix_opportunity,optimize_title_meta,517715,4.2,page_1,0.14,91-180
1,content_2dba2b1f9536,client_6208ef0f77,443434,stale_needs_refresh,refresh_content,443434,27.9,page_3_5,0.21,91-180
2,content_2c2606c5d176,client_19581e27de,347399,ctr_fix_opportunity,optimize_title_meta,347399,4.2,page_1,0.53,91-180
3,content_cb112fce36be,client_19581e27de,309910,ctr_fix_opportunity,optimize_title_meta,309910,5.6,page_1,0.16,91-180
4,content_9532f197bbc8,client_4e07408562,309192,ctr_fix_opportunity,optimize_title_meta,309192,2.0,top_3,0.87,91-180
5,content_36ff89c8214e,client_19581e27de,295097,ctr_fix_opportunity,optimize_title_meta,295097,7.3,page_1,0.05,91-180
6,content_b28d1efd668f,client_6208ef0f77,286608,stale_needs_refresh,refresh_content,286608,26.2,page_3_5,0.06,91-180
7,content_813e88069237,client_6208ef0f77,233561,stale_needs_refresh,refresh_content,233561,26.2,page_3_5,0.06,91-180
8,content_c21024970297,client_19581e27de,211366,ctr_fix_opportunity,optimize_title_meta,211366,5.1,page_1,0.41,91-180
9,content_c8e9d6ab9013,client_19581e27de,208678,ctr_fix_opportunity,optimize_title_meta,208678,9.7,page_1,0.00,91-180


## 4. Weak picks + leakage check

**Weakest picks, in order:**
- **#2 and #6** (`content_2dba2b1f9536`, `content_36ff89c8214e`) both carry `is_declining_label = 0` -- flagged by the rule but *not* actually declining by the observed 30-day trend. The rule never sees the label (that's the point of a baseline), so this is exactly the kind of miss precision@K would catch later -- not a bug, just the honest cost of a two-signal rule.
- **#7, #8, and (from the fuller queue) a third row** share one client and near-identical `avg_position` (26.2) -- a strong sign this is one templated content type or one shared technical issue at that client, not three independent refresh opportunities. A smarter v2 rule would dedupe or cap picks per client.
- **#5 and #10** lean on tier-average CTR comparisons in tiers (`top_3`, the `page_1`/`striking` boundary) that Signal B already flagged as volume-thin or boundary-sensitive -- the comparison bar itself is shakier there than in the middle of the distribution.

**Leakage check:** the score and reason-code logic use only `freshness_tier` (from `days_since_last_update`), `impressions_90d`, `position_tier` (from `avg_position`), and `ctr` -- all trailing-90-day, already-observed columns. `trend_direction`, `trend_pct`, and `is_declining_label` are never read inside the scoring cell above; I only pulled `is_declining_label` into this section afterward, read-only, to sanity-check picks #2 and #6 by eye. No future-window column and no product-side flag (`health_score`, `priority_score` -- not shipped in this dataset anyway) entered the rule.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

# Confirms none of the label-source columns were used as inputs to score/reason_code/action
forbidden = {"trend_direction", "trend_pct", "is_declining_label"}
used_in_rule = {"freshness_tier", "impressions_90d", "position_tier", "avg_position", "ctr"}
print("Any overlap between rule inputs and label-source columns?", used_in_rule & forbidden)

# The client-clustering weak pick, made concrete
dupe_check = queue.head(15).groupby("client_id").size().sort_values(ascending=False)
print("\nHow many of the top 15 rows come from the same client:")
print(dupe_check)

Any overlap between rule inputs and label-source columns? set()

How many of the top 15 rows come from the same client:
client_id
client_19581e27de    8
client_6208ef0f77    5
client_4e07408562    2
dtype: int64


## Self-check

Before I submit, confirm each line honestly:

- [x] Every section above is filled -- markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere (client/content IDs are pseudonyms already shipped in the dataset)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.